In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""


In [3]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [4]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client
)

In [5]:
result = agent.loop('how can I create evidently dahsbord?')

In [6]:
print(result.last_message)

To create a dashboard, start by adding evaluation reports to your project since these serve as the data source for visualizations.

### Steps to Create a Dashboard

1. **Create a Dashboard**:
   - **Enter Edit Mode**: Go to the dashboard and enter "Edit" mode by clicking the option in the top right corner.
   - **Add a Tab**: Click the plus sign to add a new tab. You can create a custom tab by selecting "empty" and entering a name.

   **Code Example**: If using the Python API, you can add a new tab with:
   ```python
   project.dashboard.add_tab("Another Tab")
   ```

2. **Add Panels**:
   - Once in edit mode, click "Add Panel" to start creating visualizations.
   - Configuration options will appear, allowing you to set up your panel type (text, counter, pie chart, etc.) and select metrics from your reports.
   - Review settings and click "Save", ensuring you select the tab where the panel will reside.

   **Key Panel Types**:
   - **Text Panels** for titles.
   - **Counter Panels** t

In [7]:
result.cost.total_cost

Decimal('0.00290445')

In [8]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)


result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [9]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [ ]:
result = agent.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


In [1]:
result.cost.total_cost

NameError: name 'result' is not defined